# Run a Yahoo Fantasy league collection

This notebook loads credentials from the project's local `.env` file itself. It never prints credential values.

Run the cells in order. Start with the connectivity check, then collect one week. The optional backfill cell remains disabled until you explicitly enable it.

In [1]:
from pathlib import Path
import sys

# This installs into the exact Python interpreter selected for this notebook.
INSTALL_CANDIDATES = (Path.cwd().resolve(), Path.cwd().resolve() / "yahoo-fantasy-data")
INSTALL_DIR = next((path for path in INSTALL_CANDIDATES if (path / "pyproject.toml").is_file()), None)
if INSTALL_DIR is None:
    raise RuntimeError("Open this notebook from the repository root or yahoo-fantasy-data directory.")

print(f"Installing project dependencies for notebook Python: {sys.executable}")
%pip install -e {INSTALL_DIR}

Installing project dependencies for notebook Python: /home/deck/Workspaces/Fantasy_football/Fantasy_football_coding/.venv/bin/python
Obtaining file:///home/deck/Workspaces/Fantasy_football/Fantasy_football_coding/yahoo-fantasy-data
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for yahoo-fantasy-data (pyproject.toml) ... done
  Created wheel for yahoo-fantasy-data: filename=yahoo_fantasy_data-0.1.0-0.editable-py3-none-any.whl size=3786 sha256=9b0ffe4e1dc62e60c4a4c206444feb71bd1b1117f91e70afc32517cfef87672d
  Stored in directory: /tmp/pip-ephem-wheel-cache-fd_quhql/wheels/fd/c8/5c/ad24f2d4412eef7f0ed9a0b43003a29b4e7c3797b1b335547e
Successfully built yahoo-fantasy-data
  Attempting uninstall: yahoo-fantasy-data
    Found existing installation: yahoo-fantasy-data 0.1.0
    Uninstalling yahoo-fantasy-data-0.1.0:

In [6]:
from pathlib import Path
import sys

from dotenv import load_dotenv

# Supports opening this notebook from either the repository root or this project directory.
CANDIDATES = (Path.cwd().resolve(), Path.cwd().resolve() / 'yahoo-fantasy-data')
PROJECT_DIR = next((path for path in CANDIDATES if (path / 'pyproject.toml').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('Open this notebook from the repository root or yahoo-fantasy-data directory.')

ENV_FILE = PROJECT_DIR / '.env'
if not ENV_FILE.is_file():
    raise FileNotFoundError(f'Missing {ENV_FILE}. Copy .env.example to .env and fill it in.')

# Explicit path avoids relying on VS Code terminal environment injection.
load_dotenv(ENV_FILE, override=False)

# Lets the notebook run from a source checkout; an editable install also works.
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from yahoo_fantasy_data.cli import connectivity_report
from yahoo_fantasy_data.config import load_settings
from yahoo_fantasy_data.yahoo import backfill_season, collect_week

settings = load_settings(data_dir=PROJECT_DIR / 'data')
print(f'Loaded configuration from {ENV_FILE.name}; data will be written to {settings.data_dir}')
print(f'OAuth credentials configured: {settings.oauth_configured}')

Loaded configuration from .env; data will be written to /home/deck/Workspaces/Fantasy_football/Fantasy_football_coding/yahoo-fantasy-data/data
OAuth credentials configured: False


In [ ]:
# Set these values for the Yahoo league and week you want to retrieve.
SEASON = 2025
# LEAGUE_ID = "707737"  # Numeric part only; e.g. 461.l.123456
# LEAGUE_NICKNAME = "PHFFL_A"  # Change this to the folder name you want.

# CFFL_A
# LEAGUE_ID = "134317"  # Numeric part only; e.g. 461.l.123456
# LEAGUE_NICKNAME = "CFFL_A"  # Change this to the folder name you want.

# CFFL_B
LEAGUE_ID = "918145"  # Numeric part only; e.g. 461.l.123456
LEAGUE_NICKNAME = "CFFL_B"  # Change this to the folder name you want.
WEEK = 5

# Ferda
# LEAGUE_ID = "889216"  # Numeric part only; e.g. 461.l.123456
# LEAGUE_NICKNAME = "Ferda"  # Change this to the folder name you want.

assert SEASON >= 2000
if not LEAGUE_ID.isdigit():
    raise ValueError("Use only the numeric league ID, not the full game.l.league key.")
assert WEEK >= 1

from yahoo_fantasy_data.config import storage_league_name

print(f"Data folder: {settings.data_dir / storage_league_name(LEAGUE_NICKNAME, LEAGUE_ID) / str(SEASON)}")

Data folder: /home/deck/Workspaces/Fantasy_football/Fantasy_football_coding/yahoo-fantasy-data/data/CFFL_B/2025


## 1. Connectivity check (no files written)

This reports which Yahoo endpoints currently permit this league/week.

In [5]:
report = connectivity_report(SEASON, LEAGUE_ID, WEEK)
report

{'league': '461.l.918145',
 'league_type': 'public',
 'official_api': {'reachable': False, 'oauth_required': True},
 'public_internal_api': {'reachable': True, 'anonymous_access': True},
 'players': {'available': True,
  'week_requested': 5,
  'records_returned': 1250,
  'sample': [{'season': 2025,
    'week': 5,
    'game_id': '461',
    'league_id': '918145',
    'league_key': '461.l.918145',
    'bye_week': None,
    'bye_weeks_week': '5',
    'display_position': 'DEF',
    'editorial_player_key': 'nfl.p.100001',
    'editorial_team_abbr': 'Atl',
    'editorial_team_full_name': 'Atlanta Falcons',
    'editorial_team_key': 'nfl.t.1',
    'editorial_team_url': 'https://sports.yahoo.com/nfl/teams/atlanta/',
    'eligible_positions': {'position': 'DEF'},
    'eligible_positions_0_position': nan,
    'eligible_positions_1_position': nan,
    'eligible_positions_2_position': nan,
    'eligible_positions_position': 'DEF',
    'eligible_positions_to_add_0_position': nan,
    'eligible_posit

## 2. Collect one week

Successful datasets are saved as compressed CSV snapshots. A Yahoo endpoint that requires authentication or otherwise fails does not prevent the other datasets from being saved.

In [5]:
statuses = collect_week(SEASON, LEAGUE_ID, WEEK, settings=settings, league_nickname=LEAGUE_NICKNAME)
statuses

{'player_data': 'written',
 'projection_data': 'written',
 'team_data': 'written',
 'schedule': 'written',
 'draft': 'written',
 'league_settings': 'written'}

In [6]:
# Review the snapshots that now exist for this league and season.
snapshot_dir = settings.data_dir / storage_league_name(settings.league_nickname, LEAGUE_ID) / str(SEASON)
sorted(path.relative_to(snapshot_dir).as_posix() for path in snapshot_dir.rglob('*') if path.is_file())

[]

In [8]:
import pandas as pd
df = pd.read_csv(r"/home/deck/Workspaces/Fantasy_football/Fantasy_football_coding/yahoo-fantasy-data/data/PHFFL_A/2025/schedule/schedule_week_5.csv.gz")
df

,team_key,week_1,week_10,week_11,week_12,week_13,week_14,week_2,week_3,week_4,week_5,week_6,week_7,week_8,week_9
0,461.l.707737.t.1,461.l.707737.t.14,461.l.707737.t.2,461.l.707737.t.3,461.l.707737.t.13,461.l.707737.t.15,461.l.707737.t.6,461.l.707737.t.15,461.l.707737.t.11,461.l.707737.t.12,461.l.707737.t.5,461.l.707737.t.16,461.l.707737.t.6,461.l.707737.t.14,461.l.707737.t.9
1,461.l.707737.t.10,461.l.707737.t.7,461.l.707737.t.11,461.l.707737.t.12,461.l.707737.t.5,461.l.707737.t.8,461.l.707737.t.4,461.l.707737.t.8,461.l.707737.t.2,461.l.707737.t.3,461.l.707737.t.13,461.l.707737.t.9,461.l.707737.t.4,461.l.707737.t.7,461.l.707737.t.16
2,461.l.707737.t.11,461.l.707737.t.16,461.l.707737.t.10,461.l.707737.t.4,461.l.707737.t.8,461.l.707737.t.5,461.l.707737.t.12,461.l.707737.t.5,461.l.707737.t.1,461.l.707737.t.14,461.l.707737.t.15,461.l.707737.t.6,461.l.707737.t.12,461.l.707737.t.16,461.l.707737.t.7
3,461.l.707737.t.12,461.l.707737.t.5,461.l.707737.t.7,461.l.707737.t.10,461.l.707737.t.4,461.l.707737.t.16,461.l.707737.t.11,461.l.707737.t.16,461.l.707737.t.6,461.l.707737.t.1,461.l.707737.t.14,461.l.707737.t.15,461.l.707737.t.11,461.l.707737.t.5,461.l.707737.t.8
4,461.l.707737.t.13,461.l.707737.t.3,461.l.707737.t.15,461.l.707737.t.14,461.l.707737.t.1,461.l.707737.t.2,461.l.707737.t.9,461.l.707737.t.2,461.l.707737.t.8,461.l.707737.t.4,461.l.707737.t.10,461.l.707737.t.7,461.l.707737.t.9,461.l.707737.t.3,461.l.707737.t.6
5,461.l.707737.t.14,461.l.707737.t.1,461.l.707737.t.3,461.l.707737.t.13,461.l.707737.t.9,461.l.707737.t.6,461.l.707737.t.15,461.l.707737.t.6,461.l.707737.t.16,461.l.707737.t.11,461.l.707737.t.12,461.l.707737.t.5,461.l.707737.t.15,461.l.707737.t.1,461.l.707737.t.2
6,461.l.707737.t.15,461.l.707737.t.6,461.l.707737.t.13,461.l.707737.t.9,461.l.707737.t.2,461.l.707737.t.1,461.l.707737.t.14,461.l.707737.t.1,461.l.707737.t.5,461.l.707737.t.16,461.l.707737.t.11,461.l.707737.t.12,461.l.707737.t.14,461.l.707737.t.6,461.l.707737.t.3
7,461.l.707737.t.16,461.l.707737.t.11,461.l.707737.t.4,461.l.707737.t.8,461.l.707737.t.7,461.l.707737.t.12,461.l.707737.t.5,461.l.707737.t.12,461.l.707737.t.14,461.l.707737.t.15,461.l.707737.t.6,461.l.707737.t.1,461.l.707737.t.5,461.l.707737.t.11,461.l.707737.t.10
8,461.l.707737.t.2,461.l.707737.t.9,461.l.707737.t.1,461.l.707737.t.6,461.l.707737.t.15,461.l.707737.t.13,461.l.707737.t.3,461.l.707737.t.13,461.l.707737.t.10,461.l.707737.t.7,461.l.707737.t.8,461.l.707737.t.4,461.l.707737.t.3,461.l.707737.t.9,461.l.707737.t.14
9,461.l.707737.t.3,461.l.707737.t.13,461.l.707737.t.14,461.l.707737.t.1,461.l.707737.t.6,461.l.707737.t.9,461.l.707737.t.2,461.l.707737.t.9,461.l.707737.t.4,461.l.707737.t.10,461.l.707737.t.7,461.l.707737.t.8,461.l.707737.t.2,461.l.707737.t.13,461.l.707737.t.15


## 3. Optional: backfill a whole season

Change `RUN_BACKFILL` to `True` only when you are ready. Leave `END_WEEK = None` to use the league's reported end week, or set a specific final week. Existing snapshots are skipped unless `OVERWRITE` is enabled.

In [8]:
RUN_BACKFILL = True
START_WEEK = 1
END_WEEK = None
OVERWRITE = False

if RUN_BACKFILL:
    backfill_statuses = backfill_season(
        SEASON, LEAGUE_ID, START_WEEK, END_WEEK, OVERWRITE, settings=settings, league_nickname=LEAGUE_NICKNAME
    )
    backfill_statuses
else:
    print('Backfill is disabled. Set RUN_BACKFILL = True to run it.')

YahooAPIError: Yahoo request failed for games: HTTPSConnectionPool(host='pub-api-ro.fantasysports.yahoo.com', port=443): Max retries exceeded with url: /fantasy/v2/games?format=json&game_codes=nfl&seasons=2025 (Caused by ResponseError('too many 999 error responses'))